## Imports

In [ ]:
from IPython.display import display, Markdown
from openai import OpenAI
from top_secret import my_sk

from markdown import markdown
from weasyprint import HTML

## Input Resume & JD

In [ ]:
# open and read the markdown file
with open("resumes/resume.md", "r", encoding="utf-8") as file:
    resume_string = file.read()

# input job description
jd_string = input()

## Construct Prompt

In [ ]:
prompt_template = lambda resume_string, jd_string : f"""
You are a professional resume optimization expert specializing in tailoring \
resumes to specific job descriptions. Your goal is to optimize my resume and \
provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:  
   - Prioritize experiences, skills, and achievements **most relevant to the \
job description**.  
   - Remove or de-emphasize irrelevant details to ensure a **concise** and \
**targeted** resume.
   - Limit work experience section to 2-3 most relevant roles
   - Limit bullet points under each role to 2-3 most relevant impacts

2. **Action-Driven Results**:  
   - Use **strong action verbs** and **quantifiable results** (e.g., \
percentages, revenue, efficiency improvements) to highlight impact.  

3. **Keyword Optimization**:  
   - Integrate **keywords** and phrases from the job description naturally to \
optimize for ATS (Applicant Tracking Systems).  

4. **Additional Suggestions** *(If Gaps Exist)*:  
   - If the resume does not fully align with the job description, suggest:  
     1. **Additional technical or soft skills** that I could add to make my \
profile stronger.  
     2. **Certifications or courses** I could pursue to bridge the gap.  
     3. **Project ideas or experiences** that would better align with the role.  

5. **Formatting**:  
   - Output the tailored resume in **clean Markdown format**.  
   - Include an **"Additional Suggestions"** section at the end with \
actionable improvement recommendations.  

---

### Input:
- **My resume**:  
{resume_string}

- **The job description**:  
{jd_string}

---

### Output:  
1. **Tailored Resume**:  
   - A resume in **Markdown format** that emphasizes relevant experience, \
skills, and achievements.  
   - Incorporates job description **keywords** to optimize for ATS.  
   - Uses strong language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:  
   - List **skills** that could strengthen alignment with the role.  
   - Recommend **certifications or courses** to pursue.  
   - Suggest **specific projects or experiences** to develop.
"""

## Make API Call

In [ ]:
# create prompt
prompt = prompt_template(resume_string, jd_string)

# setup api client
client = OpenAI(api_key=my_sk)

# make api call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Expert resume writer"},
        {"role": "user", "content": prompt}
    ], 
    temperature = 0.7
)

# extract response
response_string = response.choices[0].message.content

## Save New Resume

In [ ]:
# separate new resume from improvement suggestions
response_list = response_string.split("## Additional Suggestions")

In [ ]:
# Convert the markdown output to HTML using the markdown library. Then, convert the HTML to a PDF using weasyprint

# save as PDF
output_pdf_file = "resumes/resume_new.pdf"

# Convert Markdown to HTML
html_content = markdown(response_list[0])

# Convert HTML to PDF and save
HTML(string=html_content).write_pdf(output_pdf_file, 
                                    stylesheets=['resumes/style.css'])

## Improvement suggestions

In [ ]:
display(Markdown(response_list[1]))

## Create a Gradio UI

In [ ]:
import gradio as gr
from functions import *

## UI

In [ ]:
with gr.Blocks() as app:
    # create header and app description
    gr.Markdown("# Resume Optimizer 📄")
    gr.Markdown("Upload your resume, paste the job description, and get actionable insights!")

    # gather inputs
    with gr.Row():
        resume_input = gr.File(label="Upload Your Resume (.md)")    
        jd_input = gr.Textbox(label="Paste the Job Description Here", lines=9, interactive=True, placeholder="Paste job description...")
    run_button = gr.Button("Optimize Resume 🤖")

    # display outputs
    output_resume_md = gr.Markdown(label="New Resume")
    output_suggestions = gr.Markdown(label="Suggestions")

    # editing results
    output_resume = gr.Textbox(label="Edit resume and export!", interactive=True)
    export_button = gr.Button("Export Resume as PDF 🚀")
    export_result = gr.Markdown(label="Export Result")
    
    # Event binding
    run_button.click(process_resume, inputs=[resume_input, jd_input], outputs=[output_resume_md, output_resume, output_suggestions])
    export_button.click(export_resume, inputs=[output_resume], outputs=[export_result])

# Launch the app
app.launch()

In [ ]:
* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.